# Lab 08 — Caching and Reuse

**Tujuan:** Menunjukkan cache lazy, materialisasi, dan tradeoff penggunaan memori.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 08")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

In [ ]:
df = spark.range(0, 2_000_000, 1, 4)
data = df.withColumn("group_id", F.col("id") % 100)
result = data.groupBy("group_id").count()
result.count()

## 1. Cache does not compute immediately

In [ ]:
result.cache()
print("Cached object registered; action has not been called yet")

## 2. Materialize and reuse

Buka Spark UI → Storage setelah materialisasi.

In [ ]:
result.count()
result.orderBy("group_id").show(10)
result.filter("count > 1000").count()

## 3. Release cache

In [ ]:
result.unpersist()
print("Cache released")

**Tugas:** bedakan `cache`, `persist`, dan `unpersist`. Jangan menganggap cache selalu mempercepat semua workload.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.